## Import libs

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.python.data import Dataset
from tensorflow.keras.optimizers import Adam
import seaborn as sns


from falsb4mpa.modeling.zhang.learning.multi_adv import train_loop as zhang_train
from falsb4mpa.dataset.load_data import load_data
from falsb4mpa.evaluation.evaluation import compute_predictive_metrics, compute_fair_metrics, compute_adv_metrics, compute_tradeoff, fair_evaluation, compute_intersectional_fair_metrics
from falsb4mpa.modeling.zhang.models.multi_adv import ZhangMultAdv

## Preliminaries

In [2]:
batch_size = 64
epochs = 100
learning_rate = 0.001

In [3]:
cv_seeds = [13, 29, 42]
# cv_seeds = [55, 73]
# cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'adult-mpa-bin-wout-agg'

In [5]:
x, y, a1, a2 = load_data(data_name)
raw_data = (x, y, a1, a2)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
a1dim = a1.shape[1]
a2dim = a2.shape[1]
zdim = 8
print(xdim, ydim, a1dim, a2dim, zdim)

97 1 1 1 8


## Result file

In [7]:
header = [
    "model_name", "cv_seed", 
    "clas_acc", "f1-micro", "f1-macro",
    "a1_dp", "a1_deqodds", "a1_deqopp", 
    "a1_TN_g0", "a1_FP_g0", "a1_FN_g0", "a1_TP_g0", "a1_TN_g1", "a1_FP_g1", "a1_FN_g1", "a1_TP_g1", 
    "a2_dp", "a2_deqodds", "a2_deqopp", 
    "a2_TN_g0", "a2_FP_g0", "a2_FN_g0", "a2_TP_g0", "a2_TN_g1", "a2_FP_g1", "a2_FN_g1", "a2_TP_g1",
    "wc_spd", "wc_aod", "wc_eod",
    "last_cosine_similarity"
]

results = []

## Testing

### DemPar

In [8]:
fairdef = "DemPar"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4DP', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-15 23:05:02.538734: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.27/0.72 | Adv1 loss/acc 0.75/0.67 | Adv2 loss/acc 0.37/0.86 | Cos Sim 0.05


2026-05-15 23:05:17.711749: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.23/0.83 | Adv1 loss/acc 0.83/0.67 | Adv2 loss/acc 0.35/0.86 | Cos Sim 0.09
> Epoch: 3 | Clf loss/acc 0.23/0.83 | Adv1 loss/acc 0.91/0.67 | Adv2 loss/acc 0.35/0.86 | Cos Sim 0.13


2026-05-15 23:05:48.170563: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 32 | Clf loss/acc 0.45/0.83 | Adv1 loss/acc 2.12/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 33 | Clf loss/acc 0.46/0.84 | Adv1 loss/acc 2.13/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 34 | Clf loss/acc 0.46/0.84 | Adv1 loss/acc 2.14/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 35 | Clf loss/acc 0.46/0.84 | Adv1 loss/acc 2.15/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 36 | Clf loss/acc 0.47/0.84 | Adv1 loss/acc 2.16/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 37 | Clf loss/acc 0.47/0.84 | Adv1 loss/acc 2.16/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 38 | Clf loss/acc 0.47/0.84 | Adv1 loss/acc 2.17/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 39 | Clf loss/acc 0.48/0.84 | Adv1 loss/acc 2.18/0.67 | Adv2 loss/acc 0.63/0.86 | Cos Sim 0.35
> Epoch: 40 | Clf loss/acc 0.48/0.84 | Adv1 loss/acc 2.18/0.67 | Adv2 loss/acc 0.63/0.86 | Cos Sim 0.35
> Epoch: 41 | Clf loss/acc 0.48/0.84 | Adv1 loss/acc 2.19/0.67 |

2026-05-15 23:20:56.875880: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 64 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.32/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 65 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.32/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 66 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.32/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 67 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.33/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 68 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.33/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 69 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.34/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 70 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.34/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 71 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.34/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 72 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.35/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 73 | Clf loss/acc 0.52/0.84 | Adv1 loss/acc 2.35/0.67 |

2026-05-15 23:37:15.433793: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 27 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.71/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 28 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.72/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 29 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.72/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 30 | Clf loss/acc 0.56/0.84 | Adv1 loss/acc 1.73/0.67 | Adv2 loss/acc 0.40/0.86 | Cos Sim 0.49
> Epoch: 31 | Clf loss/acc 0.56/0.84 | Adv1 loss/acc 1.74/0.67 | Adv2 loss/acc 0.40/0.86 | Cos Sim 0.49
> Epoch: 32 | Clf loss/acc 0.57/0.84 | Adv1 loss/acc 1.74/0.67 | Adv2 loss/acc 0.40/0.86 | Cos Sim 0.49
> Epoch: 33 | Clf loss/acc 0.57/0.84 | Adv1 loss/acc 1.75/0.67 | Adv2 loss/acc 0.40/0.86 | Cos Sim 0.49
> Epoch: 34 | Clf loss/acc 0.57/0.84 | Adv1 loss/acc 1.76/0.67 | Adv2 loss/acc 0.40/0.86 | Cos Sim 0.49
> Epoch: 35 | Clf loss/acc 0.58/0.84 | Adv1 loss/acc 1.76/0.67 | Adv2 loss/acc 0.40/0.86 | Cos Sim 0.49
> Epoch: 36 | Clf loss/acc 0.58/0.84 | Adv1 loss/acc 1.77/0.67 |

### EqOdds

In [8]:
fairdef = "EqOdds"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOdds', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-14 20:49:45.499620: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.27/0.72 | Adv1 loss/acc 0.74/0.67 | Adv2 loss/acc 0.37/0.86 | Cos Sim 0.04


2026-05-14 20:50:01.373567: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.23/0.83 | Adv1 loss/acc 0.82/0.67 | Adv2 loss/acc 0.35/0.86 | Cos Sim 0.08
> Epoch: 3 | Clf loss/acc 0.23/0.83 | Adv1 loss/acc 0.89/0.67 | Adv2 loss/acc 0.35/0.86 | Cos Sim 0.12


2026-05-14 20:50:32.599741: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 32 | Clf loss/acc 0.44/0.84 | Adv1 loss/acc 2.09/0.67 | Adv2 loss/acc 0.61/0.86 | Cos Sim 0.35
> Epoch: 33 | Clf loss/acc 0.44/0.84 | Adv1 loss/acc 2.10/0.67 | Adv2 loss/acc 0.61/0.86 | Cos Sim 0.35
> Epoch: 34 | Clf loss/acc 0.44/0.84 | Adv1 loss/acc 2.11/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 35 | Clf loss/acc 0.45/0.84 | Adv1 loss/acc 2.12/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 36 | Clf loss/acc 0.45/0.84 | Adv1 loss/acc 2.12/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 37 | Clf loss/acc 0.45/0.84 | Adv1 loss/acc 2.13/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 38 | Clf loss/acc 0.46/0.84 | Adv1 loss/acc 2.14/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 39 | Clf loss/acc 0.46/0.84 | Adv1 loss/acc 2.15/0.67 | Adv2 loss/acc 0.62/0.86 | Cos Sim 0.35
> Epoch: 40 | Clf loss/acc 0.46/0.84 | Adv1 loss/acc 2.16/0.67 | Adv2 loss/acc 0.63/0.86 | Cos Sim 0.35
> Epoch: 41 | Clf loss/acc 0.46/0.84 | Adv1 loss/acc 2.16/0.67 |

2026-05-14 21:06:04.825428: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 64 | Clf loss/acc 0.50/0.84 | Adv1 loss/acc 2.31/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 65 | Clf loss/acc 0.50/0.84 | Adv1 loss/acc 2.31/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 66 | Clf loss/acc 0.50/0.84 | Adv1 loss/acc 2.31/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 67 | Clf loss/acc 0.50/0.84 | Adv1 loss/acc 2.32/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 68 | Clf loss/acc 0.51/0.84 | Adv1 loss/acc 2.32/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 69 | Clf loss/acc 0.51/0.84 | Adv1 loss/acc 2.33/0.67 | Adv2 loss/acc 0.65/0.86 | Cos Sim 0.35
> Epoch: 70 | Clf loss/acc 0.51/0.84 | Adv1 loss/acc 2.33/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 71 | Clf loss/acc 0.51/0.84 | Adv1 loss/acc 2.33/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 72 | Clf loss/acc 0.51/0.84 | Adv1 loss/acc 2.34/0.67 | Adv2 loss/acc 0.66/0.86 | Cos Sim 0.35
> Epoch: 73 | Clf loss/acc 0.51/0.84 | Adv1 loss/acc 2.34/0.67 |

2026-05-14 21:22:49.585755: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 27 | Clf loss/acc 0.54/0.84 | Adv1 loss/acc 1.63/0.67 | Adv2 loss/acc 0.38/0.86 | Cos Sim 0.50
> Epoch: 28 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.64/0.67 | Adv2 loss/acc 0.38/0.86 | Cos Sim 0.50
> Epoch: 29 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.65/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.50
> Epoch: 30 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.66/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.50
> Epoch: 31 | Clf loss/acc 0.56/0.84 | Adv1 loss/acc 1.67/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 32 | Clf loss/acc 0.56/0.84 | Adv1 loss/acc 1.68/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 33 | Clf loss/acc 0.56/0.84 | Adv1 loss/acc 1.69/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 34 | Clf loss/acc 0.56/0.84 | Adv1 loss/acc 1.70/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 35 | Clf loss/acc 0.57/0.84 | Adv1 loss/acc 1.71/0.67 | Adv2 loss/acc 0.39/0.86 | Cos Sim 0.49
> Epoch: 36 | Clf loss/acc 0.57/0.84 | Adv1 loss/acc 1.71/0.67 |

### EqOpp

In [8]:
fairdef = "EqOpp"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOpp', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-15 23:59:43.547483: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.37/0.73 | Adv1 loss/acc 0.11/0.67 | Adv2 loss/acc 0.12/0.86 | Cos Sim 0.20


2026-05-16 00:00:01.640248: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86 | Cos Sim 0.20
> Epoch: 3 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.10/0.66 | Adv2 loss/acc 0.12/0.80 | Cos Sim 0.19


2026-05-16 00:00:37.908111: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 64 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.16/0.57 | Adv2 loss/acc 0.15/0.53 | Cos Sim 0.27
> Epoch: 65 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.57 | Adv2 loss/acc 0.15/0.54 | Cos Sim 0.26
> Epoch: 66 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.57 | Adv2 loss/acc 0.15/0.55 | Cos Sim 0.26
> Epoch: 67 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.57 | Adv2 loss/acc 0.15/0.56 | Cos Sim 0.26
> Epoch: 68 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.57 | Adv2 loss/acc 0.15/0.58 | Cos Sim 0.25
> Epoch: 69 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.58 | Adv2 loss/acc 0.15/0.59 | Cos Sim 0.25
> Epoch: 70 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.58 | Adv2 loss/acc 0.15/0.60 | Cos Sim 0.25
> Epoch: 71 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.58 | Adv2 loss/acc 0.15/0.61 | Cos Sim 0.24
> Epoch: 72 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.58 | Adv2 loss/acc 0.15/0.61 | Cos Sim 0.24
> Epoch: 73 | Clf loss/acc 0.43/0.83 | Adv1 loss/acc 0.15/0.58 |

2026-05-16 00:37:56.460421: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 27 | Clf loss/acc 0.73/0.83 | Adv1 loss/acc 0.38/0.51 | Adv2 loss/acc 0.36/0.39 | Cos Sim 0.48
> Epoch: 28 | Clf loss/acc 0.74/0.83 | Adv1 loss/acc 0.38/0.51 | Adv2 loss/acc 0.37/0.39 | Cos Sim 0.48
> Epoch: 29 | Clf loss/acc 0.75/0.83 | Adv1 loss/acc 0.38/0.52 | Adv2 loss/acc 0.37/0.40 | Cos Sim 0.47
> Epoch: 30 | Clf loss/acc 0.75/0.83 | Adv1 loss/acc 0.38/0.52 | Adv2 loss/acc 0.37/0.40 | Cos Sim 0.47
> Epoch: 31 | Clf loss/acc 0.76/0.83 | Adv1 loss/acc 0.38/0.52 | Adv2 loss/acc 0.37/0.40 | Cos Sim 0.47
> Epoch: 32 | Clf loss/acc 0.77/0.83 | Adv1 loss/acc 0.38/0.52 | Adv2 loss/acc 0.38/0.40 | Cos Sim 0.47
> Epoch: 33 | Clf loss/acc 0.77/0.83 | Adv1 loss/acc 0.38/0.52 | Adv2 loss/acc 0.38/0.40 | Cos Sim 0.47
> Epoch: 34 | Clf loss/acc 0.78/0.83 | Adv1 loss/acc 0.37/0.52 | Adv2 loss/acc 0.38/0.40 | Cos Sim 0.47
> Epoch: 35 | Clf loss/acc 0.78/0.83 | Adv1 loss/acc 0.37/0.52 | Adv2 loss/acc 0.38/0.41 | Cos Sim 0.47
> Epoch: 36 | Clf loss/acc 0.79/0.83 | Adv1 loss/acc 0.37/0.52 |

2026-05-16 01:21:56.398474: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 54 | Clf loss/acc 0.51/0.83 | Adv1 loss/acc 0.14/0.54 | Adv2 loss/acc 0.16/0.46 | Cos Sim 0.67
> Epoch: 55 | Clf loss/acc 0.51/0.83 | Adv1 loss/acc 0.14/0.54 | Adv2 loss/acc 0.16/0.47 | Cos Sim 0.67
> Epoch: 56 | Clf loss/acc 0.51/0.83 | Adv1 loss/acc 0.14/0.54 | Adv2 loss/acc 0.16/0.48 | Cos Sim 0.67
> Epoch: 57 | Clf loss/acc 0.51/0.83 | Adv1 loss/acc 0.14/0.54 | Adv2 loss/acc 0.16/0.48 | Cos Sim 0.67
> Epoch: 58 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 0.14/0.54 | Adv2 loss/acc 0.16/0.49 | Cos Sim 0.67
> Epoch: 59 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 0.14/0.55 | Adv2 loss/acc 0.16/0.50 | Cos Sim 0.67
> Epoch: 60 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 0.14/0.55 | Adv2 loss/acc 0.16/0.50 | Cos Sim 0.67
> Epoch: 61 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 0.14/0.55 | Adv2 loss/acc 0.16/0.51 | Cos Sim 0.67
> Epoch: 62 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 0.13/0.55 | Adv2 loss/acc 0.16/0.52 | Cos Sim 0.67
> Epoch: 63 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 0.13/0.55 |

## Saving into DF then CSV

In [9]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,f1-micro,f1-macro,a1_dp,a1_deqodds,a1_deqopp,a1_TN_g0,a1_FP_g0,...,a2_FN_g0,a2_TP_g0,a2_TN_g1,a2_FP_g1,a2_FN_g1,a2_TP_g1,wc_spd,wc_aod,wc_eod,last_cosine_similarity
0,MultAdvBin4EqOpp,13,0.832494,0.832494,0.769783,0.790250,0.888897,0.890860,5391.0,898.0,...,1098.0,1946.0,1570.0,43.0,151.0,151.0,0.718996,0.776488,0.701036,0.200666
1,MultAdvBin4EqOpp,29,0.830124,0.830124,0.767658,0.779472,0.866291,0.851026,5345.0,933.0,...,1057.0,1979.0,1570.0,40.0,187.0,125.0,0.707298,0.760244,0.675141,0.535751
2,MultAdvBin4EqOpp,42,0.831457,0.831457,0.773248,0.774174,0.885057,0.894828,5239.0,980.0,...,989.0,2071.0,1529.0,57.0,178.0,122.0,0.706844,0.793267,0.706441,0.660362


In [10]:
result_df.to_csv(f'../../results/{data_name}-mult_adv-{epochs}.csv')